In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/Mathernal_Risk.csv")

# Remove leading and trailing spaces from column names
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nRisk distribution:")
print(df["Status"].value_counts())

Dataset shape: (6103, 11)

Column names:
['Patient ID', 'Name', 'Age', 'Body Temperature(F)', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']

Risk distribution:
Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64


In [4]:
# Basic dataset investigation

print("Total records:", len(df))

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nUnique patient IDs:")
print(df["Patient ID"].nunique())

print("\nRisk categories:")
print(df["Status"].unique())

print("\nNumerical statistics:")
display(df.describe())

Total records: 6103

Missing values:
Patient ID                           0
Name                                 0
Age                                  0
Body Temperature(F)                  0
Heart rate(bpm)                      0
Systolic Blood Pressure(mm Hg)       0
Diastolic Blood Pressure(mm Hg)      0
BMI(kg/m 2)                          0
Blood Glucose(HbA1c)                 0
Blood Glucose(Fasting hour-mg/dl)    0
Status                               0
dtype: int64

Exact duplicate rows:
0

Unique patient IDs:
6103

Risk categories:
<StringArray>
['high risk', 'mid risk', 'low risk']
Length: 3, dtype: str

Numerical statistics:


,Patient ID,Age,Body Temperature(F),Heart rate(bpm),Systolic Blood Pressure(mm Hg),Diastolic Blood Pressure(mm Hg),BMI(kg/m 2),Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl)
count,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000,6103.000000
mean,3152.000000,26.425037,98.665574,86.100770,129.218253,87.257578,21.435581,37.904473,5.504752
std,1761.928678,6.390205,1.590983,22.627587,17.234217,7.793099,2.157060,4.400272,0.905327
min,101.000000,15.000000,39.600000,45.000000,90.000000,9.000000,14.900000,30.000000,3.500000
25%,1626.500000,22.000000,98.600000,72.000000,120.000000,82.000000,19.600000,34.000000,4.800000
50%,3152.000000,25.000000,98.600000,80.000000,128.000000,87.000000,21.300000,38.000000,5.700000
75%,4677.500000,30.000000,98.800000,91.000000,141.000000,92.000000,23.100000,41.000000,6.000000
max,6203.000000,250.000000,104.000000,150.000000,169.000000,142.000000,27.900000,50.000000,8.900000


In [7]:
# STEP 1: Risk distribution
print("RISK DISTRIBUTION")
print(df["Status"].value_counts())
print("\nPercentage:")
print(df["Status"].value_counts(normalize=True) * 100)

# STEP 2: Suspicious values
checks = {
    "Invalid Age": ~df["Age"].between(10, 60),
    "Invalid Systolic BP": ~df["Systolic Blood Pressure(mm Hg)"].between(60, 250),
    "Invalid Diastolic BP": ~df["Diastolic Blood Pressure(mm Hg)"].between(30, 150),
    "Systolic <= Diastolic": (
        df["Systolic Blood Pressure(mm Hg)"] <=
        df["Diastolic Blood Pressure(mm Hg)"]
    ),
    "Suspicious Heart Rate": ~df["Heart rate(bpm)"].between(30, 220),
    "Suspicious Temperature": ~(
        (df["Body Temperature(F)"] - 32) * 5 / 9
    ).between(34, 43)
}

print("\nDATA QUALITY REPORT")
for name, condition in checks.items():
    print(f"{name}: {condition.sum()} records")

# Total unique records with at least one issue
import pandas as pd

issues = pd.DataFrame(checks)
print("\nTotal records with any issue:",
      issues.any(axis=1).sum())

RISK DISTRIBUTION
Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64

Percentage:
Status
high risk    33.737506
mid risk     33.475340
low risk     32.787154
Name: proportion, dtype: float64

DATA QUALITY REPORT
Invalid Age: 1 records
Invalid Systolic BP: 0 records
Invalid Diastolic BP: 1 records
Systolic <= Diastolic: 19 records
Suspicious Heart Rate: 0 records
Suspicious Temperature: 3 records

Total records with any issue: 24


In [8]:
# View records that failed any plausibility check
flagged = df.loc[issues.any(axis=1)].copy()

print("Flagged records:", len(flagged))

# Do not display patient names or IDs
display(flagged.drop(
    columns=["Name", "Patient ID"],
    errors="ignore"
))

Flagged records: 24


,Age,Body Temperature(F),Heart rate(bpm),Systolic Blood Pressure(mm Hg),Diastolic Blood Pressure(mm Hg),BMI(kg/m 2),Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl),Status
55,23,100.0,68,99,99,17.6,32,5.9,high risk
134,250,98.6,76,126,85,24.1,42,5.9,low risk
221,18,98.8,79,99,101,24.5,48,7.0,high risk
287,23,98.6,99,99,99,24.6,36,7.1,high risk
1302,29,93.0,59,93,67,17.9,37,5.6,high risk
1421,20,98.6,89,99,99,25.6,40,5.6,high risk
1863,17,98.8,144,99,101,24.5,47,7.0,high risk
2049,18,98.6,104,98,101,24.5,49,7.0,high risk
2067,39,98.6,79,99,101,24.5,48,7.0,high risk
2117,20,98.6,85,99,99,17.6,36,5.6,high risk


In [9]:
glucose_cols = [
    "Blood Glucose(HbA1c)",
    "Blood Glucose(Fasting hour-mg/dl)"
]

print("GLUCOSE STATISTICS")
display(df[glucose_cols].describe())

print("\nGLUCOSE BY RISK CLASS")
display(
    df.groupby("Status")[glucose_cols]
      .agg(["min", "median", "max"])
)

GLUCOSE STATISTICS


,Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl)
count,6103.000000,6103.000000
mean,37.904473,5.504752
std,4.400272,0.905327
min,30.000000,3.500000
25%,34.000000,4.800000
50%,38.000000,5.700000
75%,41.000000,6.000000
max,50.000000,8.900000



GLUCOSE BY RISK CLASS


Blood Glucose(HbA1c)            Blood Glucose(Fasting hour-mg/dl)  \
                           min median max                               min   
Status                                                                        
high risk                   30   38.0  50                               3.5   
low risk                    30   37.0  42                               3.6   
mid risk                    30   38.0  46                               3.8   

                       
          median  max  
Status                 
high risk    5.8  8.9  
low risk     5.2  6.7  
mid risk     5.8  6.9

In [10]:
features = [
    "Age",
    "Body Temperature(F)",
    "Heart rate(bpm)",
    "Systolic Blood Pressure(mm Hg)",
    "Diastolic Blood Pressure(mm Hg)",
    "Blood Glucose(Fasting hour-mg/dl)"
]

grouped = df.groupby(features, dropna=False)["Status"]

conflicting_groups = (grouped.nunique() > 1).sum()

print("Repeated feature combinations:",
      df.duplicated(subset=features).sum())

print("Feature combinations with conflicting risk labels:",
      conflicting_groups)

Repeated feature combinations: 548
Feature combinations with conflicting risk labels: 311


In [11]:
features = [
    "Age",
    "Body Temperature(F)",
    "Heart rate(bpm)",
    "Systolic Blood Pressure(mm Hg)",
    "Diastolic Blood Pressure(mm Hg)",
    "Blood Glucose(Fasting hour-mg/dl)"
]

grouped = df.groupby(features, dropna=False)["Status"]

print("Repeated feature combinations:",
      df.duplicated(subset=features).sum())

print("Combinations with conflicting labels:",
      (grouped.nunique() > 1).sum())

Repeated feature combinations: 548
Combinations with conflicting labels: 311


In [12]:
# Check whether conflicts remain when adding BMI and HbA1c

extended_features = features + [
    "BMI(kg/m 2)",
    "Blood Glucose(HbA1c)"
]

# Check actual names first
print("Actual columns:")
print(df.columns.tolist())

# Compare conflicts
basic_conflicts = (
    df.groupby(features, dropna=False)["Status"]
    .nunique().gt(1).sum()
)

extended_conflicts = (
    df.groupby(extended_features, dropna=False)["Status"]
    .nunique().gt(1).sum()
)

print("Conflicts with 6 features:", basic_conflicts)
print("Conflicts with 8 features:", extended_conflicts)

Actual columns:
['Patient ID', 'Name', 'Age', 'Body Temperature(F)', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']
Conflicts with 6 features: 311
Conflicts with 8 features: 0


In [ ]:
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_classif

clinical_features = [
    "Age",
    "Body Temperature(F)",
    "Heart rate(bpm)",
    "Systolic Blood Pressure(mm Hg)",
    "Diastolic Blood Pressure(mm Hg)",
    "Blood Glucose(Fasting hour-mg/dl)",
    "BMI(kg/m 2)",
    "Blood Glucose(HbA1c)"
]

X = df[clinical_features]
y = df["Status"]

mi = mutual_info_classif(
    X, y,
    random_state=42
)

results = pd.DataFrame({
    "Feature": clinical_features,
    "Mutual Information": mi
}).sort_values(
    "Mutual Information",
    ascending=False
)

display(results)

,Feature,Mutual Information
3,Systolic Blood Pressure(mm Hg),0.459677
2,Heart rate(bpm),0.336341
1,Body Temperature(F),0.302898
4,Diastolic Blood Pressure(mm Hg),0.254097
0,Age,0.204353
5,Blood Glucose(Fasting hour-mg/dl),0.151260
7,Blood Glucose(HbA1c),0.113396
6,BMI(kg/m 2),0.085137


: 